# H-001 · Does OBV-Confirmed Momentum Beat Raw Momentum?

Factor test for **H-001** (equities): whether momentum filtered by On-Balance Volume agreement predicts better than price momentum alone.

- **Idea** — Keep (or weight) price momentum only when OBV trend agrees with price direction; unconfirmed moves are flipped, zeroed, or soft-weighted.
- **Claim** — OBV-confirmed momentum has stronger next-day / next-week predictive power than raw momentum on the same universe.
- **Why it might work** — Volume-backed moves suggest informed flow; unconfirmed price moves are more likely to fade.
- **Data** — Daily OHLCV long panel; compare marginal lift over raw momentum on the same rebalance schedule.

**No `normalize` kwarg:** `add_obv_momentum_factors` always stores the **raw** combined signal (return-like; magnitude retained). Soft mode still uses an internal CS pct-rank of OBV trend as a weight inside the combination — that is not a store-side CS transform of the output.

**`mode` options** (column `obv_mom_{mode}` or `obv_mom_{mode}_{L}_{S}_{W}` when multi-window):

| Mode | Behavior |
|------|----------|
| `signed` (default) | Keep momentum when price and OBV trend signs agree; **flip the sign** (same magnitude) when they disagree. |
| `strict_zero` | Keep momentum when signs agree; set to **0** when they disagree. |
| `soft` | Continuous weight: `momentum × soft_weight`, where `soft_weight` is typically `2 × CS pct-rank(OBV trend) − 1` (not a hard agree/disagree gate). |

**Multi-window:** `lookback`, `skip`, and `obv_window` each accept an `int` or a list (cartesian product). One combo → `obv_mom_{mode}`; multiple → `obv_mom_{mode}_{L}_{S}_{W}`.

**Research IS:** load `s1_factor_panel_train.parquet` (train / research IS from the panel notebook; `RESEARCH_IS_FRACTION`, default `0.70`). Do not re-split it. Full census is `s1_factor_panel_full.parquet` (not for factor IC keep/kill). For **S1 equities** factor screens, Alphalens `periods=(1, 5, 21)` on the daily panel is the intended default (other sleeves may use different periods).

Use `data.processing.s1_feature_store.add_obv_momentum_factors` rather than reimplementing the factor inline.

Evaluation uses the S1 **trade-date** panel: Alphalens pivots `open` (no `shift(-1)`); labels are open-to-open. Prior close-to-close ICs are not comparable.


## 0. Imports & Config

Resolve the repo root, configure factor params (`lookback`, `skip`, `obv_window` as int or list; `feature_subset` / `MODE` — there is no `normalize` kwarg), and import `s1_factor_panel_train.parquet`, `add_obv_momentum_factors`, and Alphalens. The train parquet is already research IS; do not calculate another cutoff here.


In [1]:
import os
import sys

import alphalens as al
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import pandas as pd

from data.processing.cleaner import forward_fill_panel
from data.processing.feature_implementation.momentum import add_raw_momentum
from data.processing.feature_implementation.utilities import cross_sectional_pct_rank
from data.processing.s1_feature_store import add_obv_momentum_factors

# Jupyter cwd is often this notebook's folder, not the repo root; walk up until we find 01_data/ingestion.
ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

TRAIN_PANEL_PATH = os.path.join(
    ROOT, "01_data", "data_files", "s1_equities", "s1_factor_panel_train.parquet"
)

TEARSHEET_DIR = os.path.join(
    ROOT, "02_research", "notebooks", "s1_equities", "factor_tests", "tearsheets"
)

# --- Window screen (edit these lists) ---
LOOKBACKS = [5, 10, 21, 56, 91, 126, 252]       # L
SKIPS = [0, 1, 5, 21]                           # S
OBV_WINDOWS = [10, 20, 63]                      # W
# Cartesian product: every (L, S, W) with L > S is tested below via the store.

# --- Fixed for this notebook (not screened) ---
MODE = "soft"  # feature_subset ID (signed | strict_zero | soft)
# add_obv_momentum_factors has no normalize kwarg (always raw combined signal)
PERIODS = (1, 5, 21)  # fixed — H-001 / S1 default; primary narrative = 5d
QUANTILES = 5
MAX_LOSS = 0.35


## 1. Data Loading

Load `s1_factor_panel_train.parquet`, which contains the daily OHLCV chronological research IS selected by `RESEARCH_IS_FRACTION` in the panel notebook. Confirm columns needed by the feature store: `date`, `ticker`, `close`, `volume`.

Do not apply another 70/30 split in this notebook.

In [2]:
panel = pd.read_parquet(TRAIN_PANEL_PATH)
required = {"date", "ticker", "open", "close", "volume", "feature_date"}
missing = required - set(panel.columns)
if missing:
    raise ValueError(f"train panel missing columns: {sorted(missing)}")

panel = panel.copy()
panel["date"] = pd.to_datetime(panel["date"])
print(f"rows={len(panel):,}  tickers={panel['ticker'].nunique():,}  "
      f"dates={panel['date'].nunique():,}  "
      f"[{panel['date'].min().date()} → {panel['date'].max().date()}]")
panel.head()


rows=289,381  tickers=100  dates=2,915  [2010-01-05 → 2021-08-03]


,date,ticker,open,high,low,close,volume,feature_date,fwd_ret_1,fwd_ret_5,fwd_ret_21
0,2010-01-05,AAPL,6.424143,6.421146,6.357683,6.406478,493729600.0,2010-01-04,-0.001025,-0.025210,-0.083271
1,2010-01-06,AAPL,6.417558,6.453779,6.383729,6.417557,601904800.0,2010-01-05,-0.012268,-0.030367,-0.101456
2,2010-01-07,AAPL,6.338825,6.443003,6.308892,6.315478,552160000.0,2010-01-06,-0.006848,-0.007745,-0.075844
3,2010-01-08,AAPL,6.295419,6.346309,6.258000,6.303801,477131200.0,2010-01-07,0.011888,0.002996,-0.066001
4,2010-01-11,AAPL,6.370258,6.346310,6.258300,6.345711,447610800.0,2010-01-08,-0.016965,-0.021005,-0.079464


## 2. Data Cleaning & Engineering

Clean the long panel as needed (missing bars, winsorize, etc.) via project cleaners. Stay in **long format**.

Point-in-time only: features at `t` use information available at or before `t`.

In [3]:
panel = forward_fill_panel(panel, columns=["close", "volume"], limit=5)
panel = panel.dropna(subset=["close", "volume"]).reset_index(drop=True)
print(f"after clean: rows={len(panel):,}  "
      f"null close={panel['close'].isna().sum()}  "
      f"null volume={panel['volume'].isna().sum()}")


after clean: rows=289,381  null close=0  null volume=0


## 3. Modeling / Signal Construction

Build both factors on the **same** cleaned panel so the comparison is fair.

### 3.1 Raw momentum (baseline)

Skip-month style: `P_{t-S} / P_{t-L} - 1` (e.g. L = 252, S = 21). Control factor for the paired test.

### 3.2 OBV-confirmed momentum (H-001)

Call `add_obv_momentum_factors` with `feature_subset=[MODE]` (raw combined signal only).

- Output is always the raw combined signal (`obv_mom_{mode}`).
- Optionally run more than one `mode` (`signed` / `strict_zero` / `soft`) side by side for sensitivity — note which is primary.


In [4]:
def add_cs_ranked_raw_momentum(
    panel: pd.DataFrame,
    *,
    lookback: int,
    skip: int,
) -> pd.DataFrame:
    """Add CS pct-ranked raw momentum as ``raw_momentum_{L}_{S}`` (Alphalens baseline)."""
    raw_col = f"_raw_momentum_tmp_{lookback}_{skip}"
    out_col = f"raw_momentum_{lookback}_{skip}"
    out = add_raw_momentum(panel, lookback=lookback, skip=skip, col=raw_col)
    out[out_col] = cross_sectional_pct_rank(out, raw_col)
    return out.drop(columns=[raw_col])


In [5]:
panel = add_obv_momentum_factors(
    panel,
    lookback=LOOKBACKS,
    skip=SKIPS,
    obv_window=OBV_WINDOWS,
    feature_subset=[MODE],
)

OBV_COLS = [c for c in panel.columns if c.startswith(f"obv_mom_{MODE}")]
unique_ls = sorted({(L, S) for L in LOOKBACKS for S in SKIPS if L > S})
RAW_COLS = []
for L, S in unique_ls:
    panel = add_cs_ranked_raw_momentum(panel, lookback=L, skip=S)
    RAW_COLS.append(f"raw_momentum_{L}_{S}")

FACTOR_COLS = OBV_COLS + RAW_COLS
print(f"OBV factors ({len(OBV_COLS)}): {OBV_COLS}")
print(f"Raw baselines ({len(RAW_COLS)}): {RAW_COLS}")


OBV factors (72): ['obv_mom_soft_5_0_10', 'obv_mom_soft_5_0_20', 'obv_mom_soft_5_0_63', 'obv_mom_soft_5_1_10', 'obv_mom_soft_5_1_20', 'obv_mom_soft_5_1_63', 'obv_mom_soft_10_0_10', 'obv_mom_soft_10_0_20', 'obv_mom_soft_10_0_63', 'obv_mom_soft_10_1_10', 'obv_mom_soft_10_1_20', 'obv_mom_soft_10_1_63', 'obv_mom_soft_10_5_10', 'obv_mom_soft_10_5_20', 'obv_mom_soft_10_5_63', 'obv_mom_soft_21_0_10', 'obv_mom_soft_21_0_20', 'obv_mom_soft_21_0_63', 'obv_mom_soft_21_1_10', 'obv_mom_soft_21_1_20', 'obv_mom_soft_21_1_63', 'obv_mom_soft_21_5_10', 'obv_mom_soft_21_5_20', 'obv_mom_soft_21_5_63', 'obv_mom_soft_56_0_10', 'obv_mom_soft_56_0_20', 'obv_mom_soft_56_0_63', 'obv_mom_soft_56_1_10', 'obv_mom_soft_56_1_20', 'obv_mom_soft_56_1_63', 'obv_mom_soft_56_5_10', 'obv_mom_soft_56_5_20', 'obv_mom_soft_56_5_63', 'obv_mom_soft_56_21_10', 'obv_mom_soft_56_21_20', 'obv_mom_soft_56_21_63', 'obv_mom_soft_91_0_10', 'obv_mom_soft_91_0_20', 'obv_mom_soft_91_0_63', 'obv_mom_soft_91_1_10', 'obv_mom_soft_91_1_20', 

## 4. Evaluation

Paired comparison of **raw momentum** vs **OBV-confirmed momentum** on identical research-IS universe and dates from `s1_factor_panel_train.parquet`. For **S1 equities**, use Alphalens `periods=(1, 5, 21)` on the daily panel (primary narrative 5d); other strategies may choose different periods. Screen window lists on this IS only.

### 4.1 Window screen summary

Factor column names encode the windows used. Lists `LOOKBACKS` × `SKIPS` × `OBV_WINDOWS` are **cartesian-producted** in `add_obv_momentum_factors`.

| Token | Meaning | Formula / role |
|-------|---------|----------------|
| `mode` | Confirmation rule | Fixed to `soft` in this notebook |
| **L** (`lookback`) | Start-price lag | Momentum uses `P_{t-S} / P_{t-L} - 1` |
| **S** (`skip`) | End-price lag | Skips the most recent S days so short-term reversal does not contaminate momentum |
| **W** (`obv_window`) | OBV trend length | `OBV_t - OBV_{t-W}` |

**Column patterns**

- OBV (multi-combo): `obv_mom_{mode}_{L}_{S}_{W}` — e.g. `obv_mom_signed_252_21_20`
- Raw baseline: `raw_momentum_{L}_{S}` — same L/S as the momentum legs; no W

Primary ranking metric below: **mean IC at 5d** (`ic_5d`).


In [6]:
def to_alphalens_prices(panel: pd.DataFrame) -> pd.DataFrame:
    """Wide open matrix for Alphalens (trade-date panel; entry at open)."""
    prices = panel.pivot(index="date", columns="ticker", values="open")
    prices.index = pd.to_datetime(prices.index)
    return prices.sort_index()


def to_alphalens_factor(panel: pd.DataFrame, col: str) -> pd.Series:
    """MultiIndex (date, ticker) factor series for Alphalens."""
    factor = panel.set_index(["date", "ticker"])[col].dropna()
    factor.index = factor.index.set_levels(
        pd.to_datetime(factor.index.levels[0]), level=0
    )
    return factor.sort_index()


def parse_factor_name(col: str) -> dict:
    """Decode ``obv_mom_{mode}_{L}_{S}_{W}`` or ``raw_momentum_{L}_{S}``."""
    if col.startswith("raw_momentum_"):
        parts = col.split("_")
        # raw_momentum_{L}_{S}
        return {"kind": "raw", "L": int(parts[-2]), "S": int(parts[-1]), "W": pd.NA}
    if col.startswith("obv_mom_"):
        parts = col.split("_")
        # obv_mom_{mode}_{L}_{S}_{W}
        return {
            "kind": "obv",
            "L": int(parts[-3]),
            "S": int(parts[-2]),
            "W": int(parts[-1]),
        }
    raise ValueError(f"unrecognized factor column: {col!r}")


def _period_label(period_index: pd.Index, period: int, position: int):
    """Match Alphalens period label ('1D', '5D', …) or fall back by position."""
    for c in (f"{period}D", f"{period}d", period, str(period)):
        if c in period_index:
            return c
    return period_index[position]


def factor_screen_metrics(
    factor: pd.Series,
    prices: pd.DataFrame,
    *,
    periods: tuple[int, ...] = PERIODS,
    quantiles: int = QUANTILES,
    max_loss: float = MAX_LOSS,
) -> dict:
    """Mean IC and Q5−Q1 mean return spread for each forward period."""
    factor_data = al.utils.get_clean_factor_and_forward_returns(
        factor=factor,
        prices=prices,
        quantiles=quantiles,
        periods=periods,
        max_loss=max_loss,
    )
    mean_ic = al.performance.mean_information_coefficient(factor_data)
    mean_ret, _ = al.performance.mean_return_by_quantile(factor_data, demeaned=True)

    row = {}
    for i, p in enumerate(periods):
        ic_key = _period_label(mean_ic.index, p, i)
        ret_key = _period_label(mean_ret.columns, p, i)
        row[f"ic_{p}d"] = float(mean_ic.loc[ic_key])
        q_hi, q_lo = mean_ret.index.max(), mean_ret.index.min()
        row[f"spread_{p}d"] = float(
            mean_ret.loc[q_hi, ret_key] - mean_ret.loc[q_lo, ret_key]
        )
    return row


In [7]:
prices = to_alphalens_prices(panel)

rows = []
for col in FACTOR_COLS:
    meta = parse_factor_name(col)
    metrics = factor_screen_metrics(to_alphalens_factor(panel, col), prices)
    rows.append({"factor": col, **meta, **metrics})

summary = (
    pd.DataFrame(rows)
    .sort_values("ic_5d", ascending=False)
    .reset_index(drop=True)
)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.4f}".format)
summary


c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change(period)
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change(period)
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any n

,factor,kind,L,S,W,ic_1d,spread_1d,ic_5d,spread_5d,ic_21d,spread_21d
0,raw_momentum_252_5,raw,252,5,<NA>,0.0201,0.0003,0.0243,0.0010,0.0265,0.0026
1,raw_momentum_252_1,raw,252,1,<NA>,0.0198,0.0002,0.0239,0.0009,0.0270,0.0027
2,raw_momentum_252_0,raw,252,0,<NA>,0.0188,0.0002,0.0232,0.0008,0.0268,0.0028
3,raw_momentum_252_21,raw,252,21,<NA>,0.0188,0.0002,0.0228,0.0009,0.0251,0.0024
4,raw_momentum_126_1,raw,126,1,<NA>,0.0165,0.0002,0.0206,0.0012,0.0291,0.0049
5,raw_momentum_126_0,raw,126,0,<NA>,0.0158,0.0003,0.0203,0.0012,0.0293,0.0049
6,raw_momentum_126_5,raw,126,5,<NA>,0.0168,0.0003,0.0202,0.0012,0.0278,0.0046
7,raw_momentum_126_21,raw,126,21,<NA>,0.0143,0.0002,0.0171,0.0009,0.0216,0.0036
8,raw_momentum_91_1,raw,91,1,<NA>,0.0134,0.0002,0.0131,0.0005,0.0126,0.0020
9,raw_momentum_91_5,raw,91,5,<NA>,0.0130,0.0002,0.0128,0.0005,0.0107,0.0019


### 4.2 Full tear sheet (manual OBV combo)

Review the summary table in §4.1, then set `TEAR_LOOKBACK` (L), `TEAR_SKIP` (S), and `TEAR_OBV_WINDOW` (W) in the cell below. The tear sheet runs on `obv_mom_{MODE}_{L}_{S}_{W}` for those values — nothing is auto-selected.

The tear is displayed in-notebook **and** saved as a multi-page PDF under `02_research/notebooks/s1_equities/factor_tests/tearsheets/` named `H-001_obv_mom_{mode}_{L}_{S}_{W}.pdf` (hypothesis id + factor column, including window/mode args). Re-running overwrites the same path.


In [13]:
def obv_factor_col(lookback: int, skip: int, obv_window: int, *, mode: str = MODE) -> str:
    """Column name for a multi-window OBV factor: ``obv_mom_{mode}_{L}_{S}_{W}``."""
    return f"obv_mom_{mode}_{lookback}_{skip}_{obv_window}"


def run_full_tear(
    panel: pd.DataFrame,
    factor_col: str,
    prices: pd.DataFrame,
    *,
    periods: tuple[int, ...] = PERIODS,
    quantiles: int = QUANTILES,
    max_loss: float = MAX_LOSS,
    tearsheet_dir: str = TEARSHEET_DIR,
):
    """Build factor_data, run Alphalens full tear, save figs to multi-page PDF.

    Alphalens calls plt.show() after each plot, which clears figures under Agg.
    Temporarily replace plt.show so each figure is written into the PDF before close.
    """
    if factor_col not in panel.columns:
        raise ValueError(
            f"{factor_col!r} not in panel — pick L/S/W that were screened "
            f"(available: {OBV_COLS})"
        )
    plt.close("all")
    factor_data = al.utils.get_clean_factor_and_forward_returns(
        factor=to_alphalens_factor(panel, factor_col),
        prices=prices,
        quantiles=quantiles,
        periods=periods,
        max_loss=max_loss,
    )

    os.makedirs(tearsheet_dir, exist_ok=True)
    out_path = os.path.join(tearsheet_dir, f"H-001_{factor_col}.pdf")
    pdf = PdfPages(out_path)
    n_pages = 0
    _original_show = plt.show

    def _show_and_savefig(*args, **kwargs):
        nonlocal n_pages
        for num in list(plt.get_fignums()):
            fig = plt.figure(num)
            if fig.axes:
                pdf.savefig(fig, bbox_inches="tight")
                n_pages += 1
        plt.close("all")

    plt.show = _show_and_savefig
    try:
        al.tears.create_full_tear_sheet(factor_data, long_short=True)
        _show_and_savefig()
    finally:
        plt.show = _original_show
        pdf.close()
        plt.close("all")

    print(f"Wrote {out_path} ({n_pages} pages)")
    return factor_data


In [14]:
# --- Edit after reviewing the §4.1 summary table ---
TEAR_LOOKBACK = 126       # L
TEAR_SKIP = 21            # S
TEAR_OBV_WINDOW = 20      # W

tear_col = obv_factor_col(TEAR_LOOKBACK, TEAR_SKIP, TEAR_OBV_WINDOW)
print(f"Tear sheet factor: {tear_col}")
tear_factor_data = run_full_tear(panel, tear_col, prices)


Tear sheet factor: obv_mom_soft_126_21_20


c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change(period)
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change(period)
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any n

Quantiles Statistics


,min,max,mean,std,count,count %
factor_quantile,,,,,,
1,-3.0228,-0.0067,-0.1529,0.1418,55360,20.1497
2,-0.3398,0.0146,-0.0321,0.0243,54708,19.9124
3,-0.0690,0.0722,0.0014,0.0107,54608,19.8760
4,-0.0097,0.2401,0.0360,0.0237,54708,19.9124
5,0.0072,2.3283,0.1581,0.1392,55359,20.1494


Returns Analysis


...

c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\performance.py:118: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  grouper.append(pd.Grouper(freq=by_time))
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:928: UserWarning: Skipping return periods that aren't exact multiples of days.
  warnings.warn(


Turnover Analysis


,1D,5D,21D
Quantile 1 Mean Turnover,0.1820,0.4000,0.7610
Quantile 2 Mean Turnover,0.3760,0.6090,0.7650
Quantile 3 Mean Turnover,0.3950,0.5820,0.7040
Quantile 4 Mean Turnover,0.3730,0.5960,0.7790
Quantile 5 Mean Turnover,0.1790,0.3590,0.6550


,1D,5D,21D
Mean Factor Rank Autocorrelation,0.8680,0.5890,-0.0000


Wrote c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\02_research\notebooks\factor_tests\tearsheets\H-001_obv_mom_soft_126_21_20.pdf (3 pages)


## Conclusion

A soft weighted OBV with:

TEAR_LOOKBACK = 126
TEAR_SKIP = 21
TEAR_OBV_WINDOW = 20

however standard raw momentum out performs this factor.